# Best Model Feature Importance

This notebook interprets the locked **SRM Global Linear Composite**. It does not retune or replace the selected model. OOF cross-validation is used only for performance estimation; the full-data fit is used only for coefficient and contribution interpretation.

No CSV, JSON, figure, or report files are exported by this notebook.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import add_visit_time, modelling_pair_count_table
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.feature_importance import (
    annual_feature_contributions,
    bootstrap_srm_coefficients,
    coefficient_importance_table,
    correlation_redundancy,
    domain_contributions,
    fit_locked_srm_full_data,
    infer_feature_domains,
    integrated_feature_importance,
    lofo_srm_importance,
    selected_srm_config_from_log,
    selection_stability_summary,
)
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics, interval_effect_summary
from src.reporting.fold_comparison import fold_train_test_clinical_benchmark_table
from src.models.srm_global import srm_global_loocv

set_global_seeds(DEFAULT_CONFIG.random_state)
RANDOM_SEED = DEFAULT_CONFIG.random_state
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT_PERFORMANCE = 300
N_BOOT_COEFFICIENTS = 1000
N_BOOT_LOFO = 100
CORRELATION_THRESHOLD = 0.70

pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
long_path = REPO_ROOT / "data" / "processed" / "trackfa_long.csv"
log_path = REPO_ROOT / "results" / "srm_composite_optimization_log.csv"

if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
if not log_path.exists():
    raise FileNotFoundError(f"Required SRM optimization log not found: {log_path}")

pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
pair_long_df = trackfa_pairs_to_long(pairs_df)
long_df = add_visit_time(pd.read_csv(long_path), visit_col="visit") if long_path.exists() else None
feature_groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in feature_groups.all_neuroimaging if c in pair_long_df.columns]
config = selected_srm_config_from_log(pd.read_csv(log_path))

print(f"Loaded annual paired modelling table: {pairs_path.name}")
print(f"Annual long rows: {pair_long_df.shape[0]}, pair intervals: {pair_long_df['pair_id'].nunique()}, participants: {pair_long_df['subject'].nunique()}")
print(f"MRI feature candidates available: {len(imaging_cols)}")
print(f"Coefficient bootstrap resamples requested: {N_BOOT_COEFFICIENTS}")

In [ ]:
# Fold-level clinical train/test benchmark for supervisor review.
fold_train_test_clinical_benchmarks = fold_train_test_clinical_benchmark_table(
    pair_long_df,
    pairs_df,
    imaging_cols,
    subject_col="pair_id",
    visit_col="visit",
    split_group_col="subject",
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    clinical_scales=("FARS", "SARA"),
    pair_types=("V1V2", "V2V3"),
)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
fold_train_test_clinical_benchmarks.to_csv(
    RESULTS_DIR / "best_model_feature_importance_fold_train_test_clinical_benchmarks.csv",
    index=False,
)
clinical_display_cols = [
    "fold",
    "clinical_scale",
    "train_n_subjects",
    "test_n_subjects",
    "clinical_train_n_pairs",
    "clinical_test_n_pairs",
    "clinical_train_d",
    "clinical_test_d",
    "clinical_train_minus_test_d",
]
print("Best model feature importance fold-level train/test clinical benchmarks")
display(fold_train_test_clinical_benchmarks[clinical_display_cols])


## 1. Best Model Summary

The table below locks the model configuration before interpretation. These settings come from the existing SRM optimization log and are not retuned here.

In [ ]:
locked_fit = fit_locked_srm_full_data(
    pair_long_df,
    imaging_cols,
    pair_col="pair_id",
    visit_col="visit",
    split_group_col="subject",
    selection_method=config["selection_method"],
    k=config["k"],
    ridge=config["ridge"],
    covariance_shrinkage=config["covariance_shrinkage"],
    z_clip=config["z_clip"],
)
selected_features = locked_fit["feature_names"]

summary_rows = [
    {"item": "Best model", "value": "SRM Global Linear Composite"},
    {"item": "Selected hyperparameters", "value": {k: config[k] for k in ["ridge", "covariance_shrinkage", "z_clip", "k"]}},
    {"item": "Regularisation", "value": config["regularisation"]},
    {"item": "Feature-selection method", "value": config["selection_method"]},
    {"item": "Number of selected features", "value": len(selected_features)},
    {"item": "Preprocessing", "value": "complete-case MRI rows; train/full-fit standardisation; optional z clipping from locked config"},
    {"item": "Tuning metric", "value": config["tuning_metric"]},
    {"item": "Interpretation fit", "value": "locked full-data SRM fit; not used as performance estimate"},
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print("Selected features")
display(pd.DataFrame({"feature": selected_features}))

## 2. Current OOF Performance

These values are generated from subject-level grouped OOF predictions using the locked configuration. The annual paired table is used for V1->V2 and V2->V3; V1->V3 is reported separately as cumulative 24-month sensitivity when the long visit table is available.

In [ ]:
annual_oof = srm_global_loocv(
    pair_long_df,
    imaging_cols,
    subject_col="pair_id",
    visit_col="visit",
    selection_method=config["selection_method"],
    k=config["k"],
    ridge=config["ridge"],
    covariance_shrinkage=config["covariance_shrinkage"],
    z_clip=config["z_clip"],
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    compute_ci=False,
    split_group_col="subject",
    start_visit=1,
    end_visit=2,
)
annual_intervals = adjacent_pair_interval_effect_summary(
    annual_oof["oof_df"],
    pair_col="pair_id",
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT_PERFORMANCE,
    seed=RANDOM_SEED,
)
annual_diag = annual_tuning_diagnostics(annual_intervals)

if long_df is not None:
    long_features = [f for f in selected_features if f in long_df.columns]
    cumulative_oof = srm_global_loocv(
        long_df,
        long_features,
        subject_col="subject_id",
        visit_col="visit",
        selection_method="none",
        k=len(long_features),
        ridge=config["ridge"],
        covariance_shrinkage=config["covariance_shrinkage"],
        z_clip=config["z_clip"],
        cv_n_splits=CV_N_SPLITS,
        random_seed=RANDOM_SEED,
        compute_ci=False,
        start_visit=1,
        end_visit=3,
    )
    cumulative_intervals = interval_effect_summary(
        cumulative_oof["oof_df"],
        subject_col="subject_id",
        visit_col="visit",
        score_col="score",
        time_col=None,
        intervals=[(1, 3, "V1->V3", False)],
        n_boot=N_BOOT_PERFORMANCE,
        seed=RANDOM_SEED,
    )
else:
    cumulative_intervals = pd.DataFrame()

performance_rows = [
    {"metric": "V1->V2 d_z", "value": annual_diag["dz_v1_v2"]},
    {"metric": "V2->V3 d_z", "value": annual_diag["dz_v2_v3"]},
    {"metric": "mean annual d_z", "value": annual_diag["mean_validation_annual_dz"]},
    {"metric": "|d12-d23|", "value": annual_diag["annual_interval_gap"]},
    {"metric": "P(delta>0) mean across annual intervals", "value": annual_diag["p_progression"]},
]
for _, row in annual_intervals.iterrows():
    performance_rows.append({"metric": f"{row['interval']} P(delta>0)", "value": row["p_delta_positive"]})
if not cumulative_intervals.empty:
    performance_rows.append({"metric": "V1->V3 cumulative d_z", "value": cumulative_intervals.iloc[0]["d_z"]})

print("Annual interval OOF details")
display(annual_intervals)
if not cumulative_intervals.empty:
    print("24-month cumulative OOF details")
    display(cumulative_intervals)
print("Locked-model performance summary")
display(pd.DataFrame(performance_rows))

## 3. Standardised Coefficients

The SRM score is linear: `Z = w^T X`. Coefficients below are comparable because MRI predictors are standardised before fitting. Coefficient magnitude is a conditional model weight; it is not, by itself, biological importance.

In [ ]:
coef_table = coefficient_importance_table(selected_features, locked_fit["coef"])
print("Locked SRM Global Linear coefficients")
display(coef_table)


## 4. Bootstrap Stability

Bootstrap refits resample participants, not rows. Hyperparameters are fixed to the locked model configuration. The full-data interpretation coefficient is shown alongside the bootstrap distribution.

In [ ]:
boot = bootstrap_srm_coefficients(
    pair_long_df,
    imaging_cols,
    pair_col="pair_id",
    visit_col="visit",
    split_group_col="subject",
    selection_method=config["selection_method"],
    k=config["k"],
    ridge=config["ridge"],
    covariance_shrinkage=config["covariance_shrinkage"],
    z_clip=config["z_clip"],
    n_boot=N_BOOT_COEFFICIENTS,
    random_seed=RANDOM_SEED,
    locked_feature_names=selected_features,
    full_coef=locked_fit["coef"],
)
bootstrap_table = boot["summary"].sort_values("full_data_coefficient", key=lambda s: s.abs(), ascending=False, kind="mergesort").reset_index(drop=True)
print(f"Bootstrap refits retained: {boot['n_boot_kept']} / {N_BOOT_COEFFICIENTS}")
display(bootstrap_table)


## 5. Annual Feature Contributions

For a linear score, the observed longitudinal change decomposes exactly as `delta Z = sum_j w_j delta X_j`. This section measures which features actually drive the observed annual composite change.

In [ ]:
contribution_long, contribution_table = annual_feature_contributions(locked_fit, pair_col="pair_id", visit_col="visit")
print("Annual feature contributions")
display(contribution_table)


## 6. V1->V2 vs V2->V3 Contribution Difference

This section investigates why the annual sensitivity differs between intervals using observed feature contributions only.

In [ ]:
diff_table = contribution_table[["feature", "mean_contribution_V1->V2", "mean_contribution_V2->V3", "contribution_gap"]].copy()
diff_table["absolute_difference"] = diff_table["contribution_gap"].abs()
diff_table = diff_table.sort_values("absolute_difference", ascending=False, kind="mergesort").reset_index(drop=True)
print("Annual contribution differences")
display(diff_table)

v12_top = contribution_table.reindex(contribution_table["mean_contribution_V1->V2"].abs().sort_values(ascending=False).index).head(5)["feature"].tolist()
weakens = diff_table.sort_values("contribution_gap", ascending=False).head(5)["feature"].tolist()
stable = diff_table.sort_values("absolute_difference", ascending=True).head(5)["feature"].tolist()
print("Features contributing most to the V1->V2 signal:", ", ".join(v12_top))
print("Features whose contribution weakens most in V2->V3:", ", ".join(weakens))
print("Features with relatively stable contribution across both years:", ", ".join(stable))


## 7. Leave-One-Feature-Out Performance

Each selected feature is removed, the locked SRM specification is refit with fixed hyperparameters, and the same subject-level OOF annual evaluation is repeated. Larger positive loss means the annual progression sensitivity depends more on that feature. Negative loss means removing the feature improved OOF annual performance.

In [ ]:
lofo_table = lofo_srm_importance(
    pair_long_df,
    selected_features,
    pair_col="pair_id",
    visit_col="visit",
    split_group_col="subject",
    selection_method=config["selection_method"],
    k=config["k"],
    ridge=config["ridge"],
    covariance_shrinkage=config["covariance_shrinkage"],
    z_clip=config["z_clip"],
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    n_boot=N_BOOT_LOFO,
)
print("Leave-one-feature-out annual performance loss")
display(lofo_table)


## 8. Selection Stability

Selection frequency measures robustness of selection, not biological importance. For the current locked model, `selection_method='none'`, so all selected imaging features are retained by design; Jaccard is therefore expected to be 1.0 across bootstrap refits.

In [ ]:
selection_feature_table, selection_jaccard_table = selection_stability_summary(boot["selected_feature_sets"], selected_features)
display(selection_feature_table)
display(selection_jaccard_table)

## 9. Correlation / Redundancy

Correlated MRI predictors may substitute for one another. A small coefficient is not evidence of biological irrelevance when the feature is strongly redundant with another selected predictor.

In [ ]:
corr_matrix, high_corr_pairs, redundancy_flags = correlation_redundancy(
    locked_fit["standardized_X"],
    threshold=CORRELATION_THRESHOLD,
)
print(f"High-correlation threshold: |r| > {CORRELATION_THRESHOLD}")
print("Correlation matrix")
display(corr_matrix)
print("Highly correlated feature pairs")
display(high_corr_pairs)


## 10. Domain-Level Contributions

Feature domains are derived conservatively from the existing TRACK-FA feature registry and feature names. No unsupported biological domain mapping is added.

In [ ]:
domain_map = infer_feature_domains(selected_features, feature_groups)
domain_table = domain_contributions(contribution_table, domain_map)
print("Feature-to-domain map")
display(domain_map)
print("Domain-level annual contributions")
display(domain_table)


## 11. Integrated Feature-Importance Table

This table combines the available evidence without creating an arbitrary weighted score. The qualitative label is rule-based and should be read together with the displayed evidence columns.

In [ ]:
integrated_table = integrated_feature_importance(
    coef_table,
    bootstrap_table,
    contribution_table,
    lofo_table,
    redundancy_flags,
)
display(integrated_table)
print("Label counts")
display(integrated_table["interpretation_label"].value_counts(dropna=False).rename_axis("label").reset_index(name="n_features"))